In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess
import os
import warnings
warnings.filterwarnings('ignore')

# Plot style settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

#!pip install gdown -q

# Google Drive folder ID (from the shared link)
#FOLDER_ID = "1rsWlKjhOOJYaZ40K_lm3ZBffBBebIrt9"

# Download the entire folder
#!gdown --folder {FOLDER_ID}

#!ls -lh csv_data/

DATA_DIR = "/kaggle/input/datasets/ildarrakiev/gh-archive-subset"


# Data preparation (aligned with `Data Mining Plan.md`)

**Steps 2–4:** cleaning → early-stage filter at **T = 2024-01-01** → features in windows **2023-12-02 … 2023-12-31** and **2023-11-01 … 2023-12-01** (`events_daily.csv`).


In [9]:
# ============================================================================
# Load data (minimum required for plan steps 2–4)
# ============================================================================
import pandas as pd
import numpy as np
from pathlib import Path

events_daily = pd.read_csv(f"{DATA_DIR}/events_daily.csv", parse_dates=["event_date"])
repo_meta = pd.read_csv(f"{DATA_DIR}/repo_metadata.csv", parse_dates=["repo_first_seen"])

events_daily["event_date"] = pd.to_datetime(events_daily["event_date"]).dt.tz_localize(None)
repo_meta["repo_first_seen"] = pd.to_datetime(repo_meta["repo_first_seen"], errors="coerce").dt.tz_localize(None)

print("events_daily:", events_daily.shape)
print("repo_meta:", repo_meta.shape)
print("event_date range:", events_daily["event_date"].min(), "->", events_daily["event_date"].max())


events_daily: (44706264, 6)
repo_meta: (10389095, 3)
event_date range: 2023-01-01 00:00:00 -> 2024-06-30 00:00:00


In [10]:
# ============================================================================
# Step 2 — Data cleaning (per plan)
# ============================================================================
repo_meta = repo_meta.drop_duplicates(subset=["repo_id"], keep="first")
events_daily = events_daily.drop_duplicates()

repo_meta = repo_meta.dropna(subset=["repo_id", "repo_first_seen"])
events_daily = events_daily.dropna(subset=["repo_id", "event_date", "event_type"])

repo_meta["repo_id"] = repo_meta["repo_id"].astype("int64")
events_daily["repo_id"] = events_daily["repo_id"].astype("int64")

print("Missing values — repo_meta:\n", repo_meta.isnull().sum())
print("Missing values — events_daily:\n", events_daily.isnull().sum())


Missing values — repo_meta:
 repo_id            0
repo_name          0
repo_first_seen    0
dtype: int64
Missing values — events_daily:
 event_type       0
repo_id          0
repo_name        0
event_date       0
event_count      0
unique_actors    0
dtype: int64


In [11]:
# ============================================================================
# Step 3 — Filter early-stage repositories (T = 2024-01-01)
# ============================================================================
T = pd.Timestamp("2024-01-01")
SIX_MONTHS_AGO = pd.Timestamp("2023-07-01")

events_bt = events_daily[events_daily["event_date"] < T]

stars_before_T = (
    events_bt[events_bt["event_type"] == "WatchEvent"]
    .groupby("repo_id")["event_count"]
    .sum()
    .rename("stars_before_T")
)
contributors_before_T = (
    events_bt[events_bt["event_type"].isin(["PushEvent", "PullRequestEvent"])]
    .groupby("repo_id")["unique_actors"]
    .sum()
    .rename("contributors_before_T")
)

repos = repo_meta[["repo_id", "repo_name", "repo_first_seen"]].copy()
repos = repos.merge(stars_before_T, on="repo_id", how="left")
repos = repos.merge(contributors_before_T, on="repo_id", how="left")
repos = repos.fillna(0)

early_repos = repos[
    (repos["repo_first_seen"] >= SIX_MONTHS_AGO)
    & (repos["repo_first_seen"] < T)
    & (repos["stars_before_T"] < 100)
    & (repos["contributors_before_T"] < 20)
].copy()

print(f"T = {T}  |  early-stage repositories: {len(early_repos):,}")


T = 2024-01-01 00:00:00  |  early-stage repositories: 3,254,238


In [12]:
# ============================================================================
# Step 4 — Feature engineering (calendar windows from plan)
# ============================================================================
early_ids = set(early_repos["repo_id"])
ev = events_daily[
    events_daily["repo_id"].isin(early_ids) & (events_daily["event_date"] < T)
].copy()

W_RECENT_START = pd.Timestamp("2023-12-02")
W_RECENT_END = pd.Timestamp("2023-12-31")
W_PREV_START = pd.Timestamp("2023-11-01")
W_PREV_END = pd.Timestamp("2023-12-01")


def sum_event_count(df, event_type, start, end):
    m = (
        (df["event_type"] == event_type)
        & (df["event_date"] >= start)
        & (df["event_date"] < end)
    )
    return df[m].groupby("repo_id")["event_count"].sum()


def sum_unique_actors_types(df, event_types, start, end):
    m = (
        df["event_type"].isin(event_types)
        & (df["event_date"] >= start)
        & (df["event_date"] < end)
    )
    return df[m].groupby("repo_id")["unique_actors"].sum()


features = early_repos[
    ["repo_id", "repo_name", "repo_first_seen", "stars_before_T", "contributors_before_T"]
].copy()

features = features.merge(
    sum_event_count(ev, "WatchEvent", W_RECENT_START, W_RECENT_END).rename("stars_recent"),
    on="repo_id",
    how="left",
)
features = features.merge(
    sum_event_count(ev, "WatchEvent", W_PREV_START, W_PREV_END).rename("stars_prev"),
    on="repo_id",
    how="left",
)
features = features.merge(
    sum_event_count(ev, "ForkEvent", W_RECENT_START, W_RECENT_END).rename("forks_recent"),
    on="repo_id",
    how="left",
)
features = features.merge(
    sum_event_count(ev, "ForkEvent", W_PREV_START, W_PREV_END).rename("forks_prev"),
    on="repo_id",
    how="left",
)
features = features.merge(
    sum_unique_actors_types(ev, ["PushEvent", "PullRequestEvent"], W_RECENT_START, W_RECENT_END).rename(
        "contribs_recent"
    ),
    on="repo_id",
    how="left",
)
features = features.merge(
    sum_unique_actors_types(ev, ["PushEvent", "PullRequestEvent"], W_PREV_START, W_PREV_END).rename(
        "contribs_prev"
    ),
    on="repo_id",
    how="left",
)

prs_total = (
    ev[ev["event_type"] == "PullRequestEvent"].groupby("repo_id")["event_count"].sum().rename("prs_total")
)
features = features.merge(prs_total, on="repo_id", how="left")

watchers_total = (
    ev[ev["event_type"] == "WatchEvent"].groupby("repo_id")["unique_actors"].sum().rename("watchers_total")
)
features = features.merge(watchers_total, on="repo_id", how="left")

features = features.fillna(0)

features["age_days"] = (T - features["repo_first_seen"]).dt.days

features["contributor_growth"] = np.where(
    features["contribs_prev"] > 0,
    features["contribs_recent"] / features["contribs_prev"],
    0,
)
features["fork_growth"] = np.where(
    features["forks_prev"] > 0,
    features["forks_recent"] / features["forks_prev"],
    0,
)
features["star_growth"] = np.where(
    features["stars_prev"] > 0,
    features["stars_recent"] / features["stars_prev"],
    0,
)
features["growth_acceleration"] = features["star_growth"] - np.where(features["stars_prev"] > 0, 1.0, 0.0)
features["viral_ratio"] = np.where(
    features["watchers_total"] > 0,
    features["contributors_before_T"] / features["watchers_total"],
    0,
)
features["engagement"] = np.where(
    features["stars_before_T"] > 0,
    (features["forks_recent"] + features["prs_total"]) / features["stars_before_T"],
    0,
)

FEATURE_COLS = [
    "stars_before_T",
    "contributors_before_T",
    "forks_recent",
    "age_days",
    "contributor_growth",
    "fork_growth",
    "star_growth",
    "growth_acceleration",
    "viral_ratio",
    "engagement",
]

print("features shape:", features.shape)
print(features[FEATURE_COLS].describe().T)


features shape: (3254238, 20)
                           count  mean   std   min   25%   50%    75%    max
stars_before_T        3254238.00  0.09  1.34  0.00  0.00  0.00   0.00  99.00
contributors_before_T 3254238.00  1.07  2.12  0.00  0.00  0.00   1.00  19.00
forks_recent          3254238.00  0.00  0.23  0.00  0.00  0.00   0.00 142.00
age_days              3254238.00 92.65 51.86  0.00 49.00 92.00 137.00 183.00
contributor_growth    3254238.00  0.02  0.27  0.00  0.00  0.00   0.00  18.00
fork_growth           3254238.00  0.00  0.03  0.00  0.00  0.00   0.00  23.00
star_growth           3254238.00  0.00  0.09  0.00  0.00  0.00   0.00  64.00
growth_acceleration   3254238.00 -0.01  0.11 -1.00  0.00  0.00   0.00  63.00
viral_ratio           3254238.00  0.07  0.71  0.00  0.00  0.00   0.00  19.00
engagement            3254238.00  0.02  0.64  0.00  0.00  0.00   0.00 481.00


In [13]:
# ============================================================================
# Save dataset + feature matrix X for step 5 (modeling)
# ============================================================================
# Kaggle: /kaggle/input/ is read-only — write to /kaggle/working/
import os
from pathlib import Path

_data = Path(DATA_DIR).resolve()
_p = str(_data).replace("\\", "/")
if _p.startswith("/kaggle/input"):
    out_path = Path("/kaggle/working/features_prepared_plan.csv")
elif os.environ.get("COLAB_RELEASE_TAG"):
    out_path = Path("/content/features_prepared_plan.csv")
else:
    out_path = _data.parent / "features_prepared_plan.csv"

try:
    features.to_csv(out_path, index=False)
except OSError:
    out_path = Path.cwd() / "features_prepared_plan.csv"
    features.to_csv(out_path, index=False)
    print("Preferred path is not writable — saved to current working directory.")
print(f"Saved: {out_path}  |  rows: {len(features):,}")

X = features[FEATURE_COLS]
print("X for sklearn:", X.shape)


Saved: /kaggle/working/features_prepared_plan.csv  |  rows: 3,254,238
X for sklearn: (3254238, 10)
